# Tensorboard的使用


In [2]:
from torch.utils.tensorboard import SummaryWriter  # SummaryWriter是一个类


In [5]:
import numpy as np
from PIL import Image


主要使用

```python
writer = SummaryWriter('logs')  # 括号内为所写入的文件夹

writer.add_image()
writer.add_scalar()

writer.close()
```


## 对于writer.add_scalar()的使用

用于画出一个想x - y的图像

需要传入参数，global_step对应x轴，scalar_step对应y轴


In [4]:
writer = SummaryWriter("logs")
for i in range(100):
    writer.add_scalar("y = x", i, i)  # y轴在前，x轴在后

writer.close()


运行后会生成一个logs文件夹，其中有tensorboard的二进制事件文件

确保激活tensorboard后

```bash
tensorboard --logdir=事件文件所在的文件夹名( --port=6007(或者任意的端口名))
```


```bash
source .venv/bin/activate
```


In [8]:
img_path = "/Users/daxian/deep-learning/dataset/train/ants_image/5650366_e22b7e1065.jpg"
img_PIL = Image.open(img_path)
img_array = np.array(img_PIL)

writer.add_image("test_image", img_array, 1, dataformats='HWC')
writer.close()


## add_image 的参数

```python
add_image(tag, img_tensor, global_step=None, walltime=None, dataformats="CHW")
```

参数位置和 add_scalar 一致：图表名、数据、global_step、walltime，最后多一个图像专有的 dataformats。

- **tag（字符串，必填）**：图片名字，也是 TensorBoard 里的分组名。同一个 tag 多次调用会按 step 累积，用滑块切换查看；不同含义的图要用不同 tag，建议写成 `train/input`、`val/pred` 这种分层名字。
- **img_tensor（必填）**：只能传 `torch.Tensor` 或 `numpy.ndarray`，其它类型（包括图片路径字符串）会报错。
- **global_step（int，可选，默认 None）**：第几步，对应滑块刻度，一般传 epoch 或 i。不传会全部记在 step 0，看不出先后变化。
- **walltime（float，可选，默认 None）**：时间戳，平时不用管。
- **dataformats（字符串，可选，默认 "CHW"）**：说明维度顺序，必须和数据的维度数量一致。

| 数据形状 | dataformats |
| --- | --- |
| (3, H, W) | 'CHW'（默认，可省略） |
| (H, W, 3) | 'HWC' |
| (H, W) | 'HW' |
| (N, 3, H, W) | 'NCHW'，或用 add_images |

数值范围：uint8 按 0~255 用，float 按 0~1 用，超出部分会被截断。

注意三点：维度数和 dataformats 长度不一致会报 AssertionError；float 传 0~255 会显示全白；归一化后的张量有负数，不能直接看。

## add_image 参数示例

```python
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter('/Users/daxian/deep-learning/note/logs_img')

img_path = '/Users/daxian/deep-learning/dataset/train/ants_image/0013035.jpg'
img = Image.open(img_path)

# 1) numpy：HWC + uint8，必须写 dataformats
writer.add_image('ants/numpy_HWC', np.array(img), 0, dataformats='HWC')

# 2) ToTensor 之后是 tensor：CHW + float[0,1]，用默认 dataformats
img_tensor = transforms.ToTensor()(img)
writer.add_image('ants/tensor_CHW', img_tensor, 1)

# 3) 一个 batch（NCHW）要用复数的 add_images
batch = torch.stack([img_tensor, img_tensor])
writer.add_images('ants/batch', batch, 2)

writer.close()
```

运行后查看：`tensorboard --logdir=/Users/daxian/deep-learning/note/logs_img`

小提示：路径这里统一写成绝对路径，所以不需要 `import os`。如果想让路径不写死用户名，可以改写成 `os.path.expanduser('~/deep-learning/...')`，那种写法才需要 `import os`，作用是把它展开成 `/Users/daxian/deep-learning/...`。
